In [ ]:
# CSBP441 Applied Computer Vision
# Week 2 - Assignment 2 Colab Code
#
# Topic:
#   Linear algebra for computer vision: vectors, matrices, homogeneous
#   coordinates, and geometric transformations.
#
# Purpose:
#   Students first solve small transformation problems by hand, then implement
#   the same ideas in Python/OpenCV, analyze image results, and interpret why
#   these operations matter in real computer vision systems.
#
# How to use in Google Colab:
#   1. Upload this .py file to Colab or copy the cells into a notebook.
#   2. Run from top to bottom.
#   3. Complete the hand-calculation answers before running the answer-check
#      cells.
#   4. Check the generated outputs/ folder.



# Assignment 2 - Linear Algebra and Geometric Transformations

This assignment follows the course cycle:

**hand calculation -> OpenCV/Python implementation -> result analysis -> real-system interpretation**

## Learning Goals

By the end of this assignment, you should be able to:

- represent points, vectors, images, and transformations using arrays/matrices;
- compute simple vector and matrix operations by hand;
- explain why translation needs homogeneous coordinates;
- apply scaling, rotation, translation, affine, and perspective transforms in OpenCV;
- interpret how geometric transformations are used in real computer vision systems.



## Setup in Google Colab

OpenCV, NumPy, and Matplotlib are usually already installed in Colab.
Run the installation cell if `import cv2` fails.



In [ ]:
# Uncomment and run this line in Colab only if OpenCV is missing:
# !pip install opencv-python numpy matplotlib



In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np


OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("OpenCV version:", cv2.__version__)
print("Output folder:", OUTPUT_DIR.resolve())




## Part A - Hand Calculation: Vectors

Complete these by hand first.

Given:

\[
a = \begin{bmatrix}3\\4\end{bmatrix}, \quad
b = \begin{bmatrix}6\\8\end{bmatrix}, \quad
c = \begin{bmatrix}-4\\3\end{bmatrix}
\]

Answer by hand:

1. Compute \(||a||_2\).
2. Compute \(a \cdot b\). What does the result say about the direction of \(a\) and \(b\)?
3. Compute \(a \cdot c\). What does the result say about the angle between \(a\) and \(c\)?
4. Compute the 2D distance between points \(p=(2,3)\) and \(q=(8,11)\).



In [ ]:
a = np.array([3, 4])
b = np.array([6, 8])
c = np.array([-4, 3])
p = np.array([2, 3])
q = np.array([8, 11])

print("Check after solving by hand:")
print("||a||2 =", np.linalg.norm(a))
print("a dot b =", np.dot(a, b))
print("a dot c =", np.dot(a, c))
print("distance(p, q) =", np.linalg.norm(q - p))




### Visualization A - Vectors, Direction, and Distance

The plot below shows vectors \(a\), \(b\), and \(c\) from the origin.
Notice that \(b\) points in the same direction as \(a\), while \(c\) is
perpendicular to \(a\). The second plot shows the distance from \(p\) to \(q\).



In [ ]:
def setup_coordinate_axis(ax, title, xlim=(-6, 10), ylim=(-6, 12)):
    ax.axhline(0, color="black", linewidth=1)
    ax.axvline(0, color="black", linewidth=1)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True, linestyle="--", alpha=0.4)
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("y")


fig, axes = plt.subplots(1, 2, figsize=(14, 5))

setup_coordinate_axis(axes[0], "Vector Direction and Magnitude")
for vec, label, color in [(a, "a=[3,4]", "tab:blue"), (b, "b=[6,8]", "tab:green"), (c, "c=[-4,3]", "tab:red")]:
    axes[0].arrow(0, 0, vec[0], vec[1], head_width=0.25, length_includes_head=True, color=color)
    axes[0].text(vec[0] + 0.2, vec[1] + 0.2, label, color=color, fontsize=11)

setup_coordinate_axis(axes[1], "Distance Between Two Points")
axes[1].scatter([p[0], q[0]], [p[1], q[1]], color=["tab:purple", "tab:orange"], s=80)
axes[1].plot([p[0], q[0]], [p[1], q[1]], color="tab:gray", linewidth=2)
axes[1].text(p[0] + 0.2, p[1], "p=(2,3)", fontsize=11)
axes[1].text(q[0] + 0.2, q[1], "q=(8,11)", fontsize=11)
axes[1].text(4.4, 7.3, f"distance = {np.linalg.norm(q - p):.2f}", fontsize=12)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "vector_and_distance_visualization.png", dpi=150)
plt.show()




### Visualization A2 - Dot Product as Projection

The dot product connects algebra to geometry:

- A large positive dot product means two vectors point in a similar direction.
- A zero dot product means two vectors are perpendicular.
- A negative dot product means two vectors point in mostly opposite directions.

The projection shows how much of one vector lies in the direction of another.



In [ ]:
def projection_of_u_on_v(u, v):
    """Return the vector projection of u onto v."""
    return (np.dot(u, v) / np.dot(v, v)) * v


proj_b_on_a = projection_of_u_on_v(b, a)
proj_c_on_a = projection_of_u_on_v(c, a)

cos_ab = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
cos_ac = np.dot(a, c) / (np.linalg.norm(a) * np.linalg.norm(c))

angle_ab = np.degrees(np.arccos(np.clip(cos_ab, -1, 1)))
angle_ac = np.degrees(np.arccos(np.clip(cos_ac, -1, 1)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

setup_coordinate_axis(axes[0], "a · b: Same Direction", xlim=(-1, 9), ylim=(-1, 10))
axes[0].arrow(0, 0, a[0], a[1], head_width=0.25, length_includes_head=True, color="tab:blue", label="a")
axes[0].arrow(0, 0, b[0], b[1], head_width=0.25, length_includes_head=True, color="tab:green", label="b")
axes[0].arrow(0, 0, proj_b_on_a[0], proj_b_on_a[1], head_width=0.25, length_includes_head=True, color="tab:red")
axes[0].text(a[0] + 0.2, a[1], "a", color="tab:blue", fontsize=12)
axes[0].text(b[0] + 0.2, b[1], "b", color="tab:green", fontsize=12)
axes[0].text(proj_b_on_a[0] - 1.4, proj_b_on_a[1] + 0.4, "projection of b on a", color="tab:red", fontsize=11)
axes[0].text(0.5, 8.8, f"a·b = {np.dot(a, b)}\nangle = {angle_ab:.1f} degrees", fontsize=12)

setup_coordinate_axis(axes[1], "a · c: Perpendicular", xlim=(-6, 5), ylim=(-1, 6))
axes[1].arrow(0, 0, a[0], a[1], head_width=0.25, length_includes_head=True, color="tab:blue")
axes[1].arrow(0, 0, c[0], c[1], head_width=0.25, length_includes_head=True, color="tab:orange")
axes[1].scatter([proj_c_on_a[0]], [proj_c_on_a[1]], color="tab:red", s=80)
axes[1].text(a[0] + 0.2, a[1], "a", color="tab:blue", fontsize=12)
axes[1].text(c[0] - 1.1, c[1] + 0.2, "c", color="tab:orange", fontsize=12)
axes[1].text(0.2, 0.3, "projection of c on a = 0", color="tab:red", fontsize=11)
axes[1].text(-5.5, 5.2, f"a·c = {np.dot(a, c)}\nangle = {angle_ac:.1f} degrees", fontsize=12)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "dot_product_projection_visualization.png", dpi=150)
plt.show()

print("Dot product interpretation:")
print(f"a dot b = {np.dot(a, b)}; angle = {angle_ab:.1f} degrees; b points in the same direction as a.")
print(f"a dot c = {np.dot(a, c)}; angle = {angle_ac:.1f} degrees; c is perpendicular to a.")




## Part B - Hand Calculation: 2D Transformations

Complete these by hand first.

Given point:

\[
P = \begin{bmatrix}2\\3\end{bmatrix}
\]

1. Apply scaling \(s_x=2, s_y=3\):

\[
S =
\begin{bmatrix}
2 & 0\\
0 & 3
\end{bmatrix}
\]

2. Rotate \(P=(1,0)\) by \(90^\circ\) counter-clockwise:

\[
R =
\begin{bmatrix}
0 & -1\\
1 & 0
\end{bmatrix}
\]

3. Translate \(P=(2,3)\) by \(t_x=5, t_y=-1\) using homogeneous coordinates:

\[
T =
\begin{bmatrix}
1 & 0 & 5\\
0 & 1 & -1\\
0 & 0 & 1
\end{bmatrix}
\]



In [ ]:
P = np.array([2, 3])
S = np.array([[2, 0], [0, 3]])

P_rotate = np.array([1, 0])
R90 = np.array([[0, -1], [1, 0]])

P_h = np.array([2, 3, 1])
T = np.array([[1, 0, 5], [0, 1, -1], [0, 0, 1]])

print("Check after solving by hand:")
print("S @ P =", S @ P)
print("R90 @ [1,0] =", R90 @ P_rotate)
print("T @ [2,3,1] =", T @ P_h)




### Visualization B - Point Transformations

This plot shows how the same point changes after scaling, rotation, and
homogeneous-coordinate translation.



In [ ]:
scaled_P = S @ P
rotated_P = R90 @ P_rotate
translated_P = T @ P_h

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

setup_coordinate_axis(axes[0], "Scaling: P=(2,3) -> S@P", xlim=(-1, 8), ylim=(-1, 11))
axes[0].scatter([P[0], scaled_P[0]], [P[1], scaled_P[1]], color=["tab:blue", "tab:red"], s=90)
axes[0].plot([0, P[0]], [0, P[1]], color="tab:blue", linewidth=2, label="original")
axes[0].plot([0, scaled_P[0]], [0, scaled_P[1]], color="tab:red", linewidth=2, label="scaled")
axes[0].text(P[0] + 0.15, P[1], "P=(2,3)", fontsize=11)
axes[0].text(scaled_P[0] + 0.15, scaled_P[1], f"S@P={tuple(scaled_P)}", fontsize=11)
axes[0].legend()

setup_coordinate_axis(axes[1], "Rotation: (1,0) by 90 degrees", xlim=(-2, 2), ylim=(-1, 2))
axes[1].arrow(0, 0, P_rotate[0], P_rotate[1], head_width=0.12, length_includes_head=True, color="tab:blue")
axes[1].arrow(0, 0, rotated_P[0], rotated_P[1], head_width=0.12, length_includes_head=True, color="tab:red")
axes[1].text(P_rotate[0] + 0.1, P_rotate[1], "original (1,0)", fontsize=11)
axes[1].text(rotated_P[0] + 0.1, rotated_P[1], f"rotated {tuple(rotated_P)}", fontsize=11)

setup_coordinate_axis(axes[2], "Translation Using Homogeneous Coordinates", xlim=(-1, 9), ylim=(-1, 5))
axes[2].scatter([P_h[0], translated_P[0]], [P_h[1], translated_P[1]], color=["tab:blue", "tab:red"], s=90)
axes[2].arrow(P_h[0], P_h[1], 5, -1, head_width=0.2, length_includes_head=True, color="tab:green")
axes[2].text(P_h[0] + 0.15, P_h[1], "P=(2,3)", fontsize=11)
axes[2].text(translated_P[0] + 0.15, translated_P[1], f"P'={tuple(translated_P[:2])}", fontsize=11)
axes[2].text(3.8, 2.1, "translation vector (5,-1)", color="tab:green", fontsize=11)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "point_transformations_visualization.png", dpi=150)
plt.show()




## Part C - Why Homogeneous Coordinates?

A 2x2 matrix can scale, rotate, shear, or reflect a 2D point. But it cannot
translate the point because the origin always maps to the origin:

\[
A \begin{bmatrix}0\\0\end{bmatrix}
=
\begin{bmatrix}0\\0\end{bmatrix}
\]

To write translation as matrix multiplication, we add one extra coordinate:

\[
(x,y) \rightarrow (x,y,1)
\]

Then shifting by \((t_x,t_y)\) becomes:

\[
\begin{bmatrix}
x'\\y'\\1
\end{bmatrix}
=
\begin{bmatrix}
1 & 0 & t_x\\
0 & 1 & t_y\\
0 & 0 & 1
\end{bmatrix}
\begin{bmatrix}
x\\y\\1
\end{bmatrix}
\]

**Question:** In your own words, why do we need the extra dimension?



In [ ]:
# Demonstration: a 2x2 matrix cannot move the origin away from (0,0).
origin_2d = np.array([0, 0])
A = np.array([[2, 1], [1, 2]])

origin_h = np.array([0, 0, 1])
T_demo = np.array([[1, 0, 100], [0, 1, 50], [0, 0, 1]])

print("2x2 matrix applied to origin:", A @ origin_2d)
print("3x3 homogeneous translation applied to origin:", T_demo @ origin_h)




## Part D - Create a Test Image

The image below is synthetic so everyone gets the same result. It contains:

- a grid for measuring geometric changes;
- colored shapes for visual interpretation;
- labeled points for transformation tracking.



In [ ]:
def create_test_image(width=500, height=360):
    img = np.full((height, width, 3), 245, dtype=np.uint8)

    # Grid.
    for x in range(0, width, 50):
        cv2.line(img, (x, 0), (x, height), (220, 220, 220), 1)
    for y in range(0, height, 50):
        cv2.line(img, (0, y), (width, y), (220, 220, 220), 1)

    # Coordinate axes.
    cv2.arrowedLine(img, (40, height - 40), (180, height - 40), (0, 0, 0), 2)
    cv2.arrowedLine(img, (40, height - 40), (40, height - 180), (0, 0, 0), 2)
    cv2.putText(img, "x", (190, height - 34), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)
    cv2.putText(img, "y", (32, height - 190), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)

    # Main objects.
    cv2.rectangle(img, (170, 95), (330, 245), (60, 130, 240), -1)
    cv2.circle(img, (250, 170), 45, (60, 180, 90), -1)
    cv2.line(img, (170, 95), (330, 245), (40, 40, 40), 4)
    cv2.line(img, (330, 95), (170, 245), (40, 40, 40), 4)

    # Labeled points.
    points = {
        "A": (170, 95),
        "B": (330, 95),
        "C": (330, 245),
        "D": (170, 245),
        "O": (250, 170),
    }
    for label, (x, y) in points.items():
        cv2.circle(img, (x, y), 6, (0, 0, 255), -1)
        cv2.putText(img, label, (x + 8, y - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 150), 2)

    cv2.putText(img, "CSBP441", (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (80, 80, 80), 2)
    return img, points


img, points = create_test_image()
cv2.imwrite(str(OUTPUT_DIR / "original_test_image.png"), img)

plt.figure(figsize=(7, 5))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title("Original Test Image")
plt.axis("off")
plt.show()

print("Image shape:", img.shape)
print("Tracked points:", points)




## Part E - Apply Translation in OpenCV

Translate the image by \(t_x=80\), \(t_y=40\).

Hand question before running:

If point \(A=(170,95)\), what is its new coordinate after translation?



In [ ]:
tx, ty = 80, 40
M_translate = np.float32([[1, 0, tx], [0, 1, ty]])
translated = cv2.warpAffine(img, M_translate, (img.shape[1], img.shape[0]))

cv2.imwrite(str(OUTPUT_DIR / "translated_tx80_ty40.png"), translated)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title("Original")
plt.axis("off")
plt.subplot(1, 2, 2)
plt.imshow(cv2.cvtColor(translated, cv2.COLOR_BGR2RGB))
plt.title("Translated: tx=80, ty=40")
plt.axis("off")
plt.show()

A = np.array([points["A"][0], points["A"][1], 1])
T_translate = np.array([[1, 0, tx], [0, 1, ty], [0, 0, 1]])
print("A after homogeneous translation:", T_translate @ A)




## Part F - Apply Scaling and Rotation

OpenCV rotates around a center point. Here we rotate the image around its
center by \(30^\circ\), then compare it with resizing.

Questions:

1. What happens to the image size when we use `cv2.resize` with `fx=1.5`, `fy=1.5`?
2. Why does rotation create black/empty regions near the corners?
3. Why is the rotation matrix from OpenCV 2x3 instead of 2x2?

Important Python note:

Use a decimal point: `1.5`

Do not write `1,5`. In Python, `1,5` means two separate values, not one
decimal number.



In [ ]:
scale_x = 1.5
scale_y = 1.5

scaled = cv2.resize(img, None, fx=scale_x, fy=scale_y, interpolation=cv2.INTER_LINEAR)
cv2.imwrite(str(OUTPUT_DIR / "scaled_fx1_5_fy1_5.png"), scaled)

h, w = img.shape[:2]
center = (w // 2, h // 2)
M_rotate = cv2.getRotationMatrix2D(center, angle=30, scale=1.0)
rotated = cv2.warpAffine(img, M_rotate, (w, h))
cv2.imwrite(str(OUTPUT_DIR / "rotated_30_degrees.png"), rotated)

# Important display note:
# Matplotlib normally stretches every image to fit its subplot. That can make
# the original and scaled image look the same size even when their pixel sizes
# are different. Here, the first two axes use the same coordinate limits so the
# scaled image appears physically larger.
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

max_h, max_w = scaled.shape[:2]

axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), extent=[0, w, h, 0])
axes[0].set_title(f"Original: {w} x {h}")
axes[0].set_xlim(0, max_w)
axes[0].set_ylim(max_h, 0)
axes[0].set_aspect("equal")
axes[0].grid(True, color="gray", alpha=0.25)

axes[1].imshow(cv2.cvtColor(scaled, cv2.COLOR_BGR2RGB), extent=[0, max_w, max_h, 0])
axes[1].set_title(f"Scaled: {max_w} x {max_h}")
axes[1].set_xlim(0, max_w)
axes[1].set_ylim(max_h, 0)
axes[1].set_aspect("equal")
axes[1].grid(True, color="gray", alpha=0.25)

axes[2].imshow(cv2.cvtColor(rotated, cv2.COLOR_BGR2RGB))
axes[2].set_title("Rotated: 30 degrees")
axes[2].axis("off")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "scaling_rotation_visual_comparison.png", dpi=150)
plt.show()

print("Original shape:", img.shape)
print("Scaled shape:", scaled.shape)
print("The scaled image has more pixels, but a normal subplot may shrink it visually.")
print("OpenCV rotation matrix:")
print(M_rotate)




## Part G - Transformation Order Matters

We will apply the same two operations in different order:

- Case 1: rotate then translate
- Case 2: translate then rotate

Question before running:

Do you expect the final image to be the same or different? Why?



In [ ]:
M_translation = np.array([[1, 0, 80], [0, 1, 30]], dtype=np.float32)
M_rotation = cv2.getRotationMatrix2D(center, angle=25, scale=1.0)

rotate_then_translate = cv2.warpAffine(
    cv2.warpAffine(img, M_rotation, (w, h)),
    M_translation,
    (w, h),
)

translate_then_rotate = cv2.warpAffine(
    cv2.warpAffine(img, M_translation, (w, h)),
    M_rotation,
    (w, h),
)

cv2.imwrite(str(OUTPUT_DIR / "rotate_then_translate.png"), rotate_then_translate)
cv2.imwrite(str(OUTPUT_DIR / "translate_then_rotate.png"), translate_then_rotate)

plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title("Original")
plt.axis("off")
plt.subplot(1, 3, 2)
plt.imshow(cv2.cvtColor(rotate_then_translate, cv2.COLOR_BGR2RGB))
plt.title("Rotate -> Translate")
plt.axis("off")
plt.subplot(1, 3, 3)
plt.imshow(cv2.cvtColor(translate_then_rotate, cv2.COLOR_BGR2RGB))
plt.title("Translate -> Rotate")
plt.axis("off")
plt.show()

difference = cv2.absdiff(rotate_then_translate, translate_then_rotate)
print("Mean pixel difference between the two results:", difference.mean())




## Part H - Affine Transformation

An affine transformation preserves parallel lines. We need 3 source points and
3 destination points.

Questions:

1. Which three points did we choose?
2. Are parallel grid lines still parallel after the affine transform?
3. Name one real application where affine transformation is useful.



In [ ]:
src_affine = np.float32([
    points["A"],
    points["B"],
    points["D"],
])

dst_affine = np.float32([
    [130, 120],
    [360, 80],
    [190, 275],
])

M_affine = cv2.getAffineTransform(src_affine, dst_affine)
affine = cv2.warpAffine(img, M_affine, (w, h))
cv2.imwrite(str(OUTPUT_DIR / "affine_transform.png"), affine)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title("Original")
plt.axis("off")
plt.subplot(1, 2, 2)
plt.imshow(cv2.cvtColor(affine, cv2.COLOR_BGR2RGB))
plt.title("Affine Transform")
plt.axis("off")
plt.show()

print("Affine matrix:")
print(M_affine)




## Part I - Perspective Transformation

A perspective transformation can simulate a change in viewpoint. It maps four
source points to four destination points.

Questions:

1. Why do we need four point pairs for perspective transformation?
2. Are parallel lines always preserved after perspective transformation?
3. Name one real application where perspective transformation is useful.



In [ ]:
src_perspective = np.float32([
    points["A"],
    points["B"],
    points["C"],
    points["D"],
])

dst_perspective = np.float32([
    [145, 85],
    [360, 115],
    [330, 280],
    [110, 245],
])

M_perspective = cv2.getPerspectiveTransform(src_perspective, dst_perspective)
perspective = cv2.warpPerspective(img, M_perspective, (w, h))
cv2.imwrite(str(OUTPUT_DIR / "perspective_transform.png"), perspective)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title("Original")
plt.axis("off")
plt.subplot(1, 2, 2)
plt.imshow(cv2.cvtColor(perspective, cv2.COLOR_BGR2RGB))
plt.title("Perspective Transform")
plt.axis("off")
plt.show()

print("Perspective matrix:")
print(M_perspective)




## Part J - Perspective Transformation for Document Scanning

A common real application of perspective transformation is document scanning.
When a document is photographed from an angle, its rectangular shape appears
like a tilted quadrilateral. Perspective transformation can map its four
corners back to a front-facing rectangle.

Questions:

1. Why are four document corners enough to rectify the document?
2. What would happen if the corner points are selected incorrectly?
3. How could this step help OCR, barcode reading, or form processing?



In [ ]:
def create_tilted_document_scene(width=700, height=500):
    scene = np.full((height, width, 3), 215, dtype=np.uint8)

    # Background desk/grid.
    for x in range(0, width, 50):
        cv2.line(scene, (x, 0), (x, height), (195, 195, 195), 1)
    for y in range(0, height, 50):
        cv2.line(scene, (0, y), (width, y), (195, 195, 195), 1)

    # Four visible document corners in the camera image.
    doc_corners = np.float32([
        [150, 80],   # top-left
        [555, 125],  # top-right
        [505, 420],  # bottom-right
        [105, 365],  # bottom-left
    ])

    # Draw document body.
    cv2.fillConvexPoly(scene, doc_corners.astype(np.int32), (250, 250, 245))
    cv2.polylines(scene, [doc_corners.astype(np.int32)], True, (40, 40, 40), 3)

    # Add document-like text lines using interpolated positions.
    for i in range(8):
        alpha = 0.18 + i * 0.085
        left = (1 - alpha) * doc_corners[0] + alpha * doc_corners[3]
        right = (1 - alpha) * doc_corners[1] + alpha * doc_corners[2]
        start = (0.18 * right + 0.82 * left).astype(int)
        end = (0.78 * right + 0.22 * left).astype(int)
        cv2.line(scene, tuple(start), tuple(end), (80, 80, 80), 3)

    cv2.putText(scene, "DOCUMENT", (245, 165), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (35, 35, 35), 2)
    cv2.putText(scene, "camera view", (20, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (90, 90, 90), 2)

    return scene, doc_corners


document_scene, document_src = create_tilted_document_scene()

document_width = 420
document_height = 300
document_dst = np.float32([
    [0, 0],
    [document_width - 1, 0],
    [document_width - 1, document_height - 1],
    [0, document_height - 1],
])

M_document = cv2.getPerspectiveTransform(document_src, document_dst)
document_rectified = cv2.warpPerspective(document_scene, M_document, (document_width, document_height))

document_scene_points = document_scene.copy()
document_rectified_points = document_rectified.copy()

corner_labels = ["P1", "P2", "P3", "P4"]
for label, (x, y) in zip(corner_labels, document_src.astype(int)):
    cv2.circle(document_scene_points, (x, y), 9, (0, 0, 255), -1)
    cv2.circle(document_scene_points, (x, y), 13, (255, 255, 255), 2)
    cv2.putText(
        document_scene_points,
        f"{label} ({x},{y})",
        (x + 12, y - 10),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (0, 0, 180),
        2,
    )

for label, (x, y) in zip(corner_labels, document_dst.astype(int)):
    # Pull labels slightly inside the output image so they are readable.
    label_x = min(max(x + 10, 10), document_width - 115)
    label_y = min(max(y + 24, 24), document_height - 12)
    cv2.circle(document_rectified_points, (x, y), 8, (0, 0, 255), -1)
    cv2.circle(document_rectified_points, (x, y), 12, (255, 255, 255), 2)
    cv2.putText(
        document_rectified_points,
        f"{label}",
        (label_x, label_y),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        (0, 0, 180),
        2,
    )

cv2.imwrite(str(OUTPUT_DIR / "document_camera_view.png"), document_scene)
cv2.imwrite(str(OUTPUT_DIR / "document_rectified_scan.png"), document_rectified)
cv2.imwrite(str(OUTPUT_DIR / "document_camera_view_points.png"), document_scene_points)
cv2.imwrite(str(OUTPUT_DIR / "document_rectified_scan_points.png"), document_rectified_points)

plt.figure(figsize=(13, 5))
plt.subplot(1, 2, 1)
plt.imshow(cv2.cvtColor(document_scene_points, cv2.COLOR_BGR2RGB))
plt.title("Camera View: Selected Corners")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(cv2.cvtColor(document_rectified_points, cv2.COLOR_BGR2RGB))
plt.title("Rectified Scan: Destination Corners")
plt.axis("off")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "document_scanning_perspective_example.png", dpi=150)
plt.show()

print("Document perspective matrix:")
print(M_document)
print("Saved:", OUTPUT_DIR / "document_scanning_perspective_example.png")




## Part K - Visual Comparison of All Image Transformations

Use this panel to compare the output of every OpenCV operation side by side.
Focus on shape changes, empty regions, parallel lines, and viewpoint changes.



In [ ]:
comparison_images = [
    ("Original", img),
    ("Translated", translated),
    ("Scaled", scaled),
    ("Rotated", rotated),
    ("Rotate -> Translate", rotate_then_translate),
    ("Translate -> Rotate", translate_then_rotate),
    ("Affine", affine),
    ("Perspective", perspective),
    ("Document Corners", document_scene_points),
    ("Rectified Corners", document_rectified_points),
]

plt.figure(figsize=(16, 12))
for i, (title_text, image_bgr) in enumerate(comparison_images, start=1):
    plt.subplot(3, 4, i)
    plt.imshow(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
    plt.title(title_text)
    plt.axis("off")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "all_transformations_comparison.png", dpi=150)
plt.show()

print("Saved side-by-side comparison:", OUTPUT_DIR / "all_transformations_comparison.png")




## Part L - Student Analysis Questions

Answer these in your submitted report.

1. Which transformation was easiest to understand by hand? Why?
2. Which OpenCV result changed the image most strongly? Explain using the output image.
3. What information is lost when parts of the image move outside the canvas?
4. Why are homogeneous coordinates important for translation?
5. Compare affine and perspective transformation:
   - What do they have in common?
   - What is different?
   - Which one is more suitable for document scanning or license plate rectification?
6. Give one real computer vision system that uses geometric transformations.
   Explain the input, transformation, output, and decision/action.

## Submission Checklist

Submit:

- hand calculations for Parts A, B, and C;
- Colab code/output screenshots or exported notebook;
- saved output images from the `outputs/` folder;
- answers to the analysis questions in Part L;
- a short real-system interpretation paragraph.



In [ ]:
print("Generated output files:")
for path in sorted(OUTPUT_DIR.glob("*")):
    print("-", path)
